In [1]:
import lamindb as ln
import pysam
import pandas as pd
import tempfile
from concurrent.futures import ThreadPoolExecutor, as_completed

ln.track("dqmJBREmxYSV", project="Lakehouse benchmarks v1")

upath = ln.UPath("s3://1000genomes-dragen")
schema = ln.Schema.get(name="1000 Genomes CNV VCF")
cnv_label = ln.ULabel(name="CNV").save()

def read_vcf(vcf_uri):
    """Stream VCF directly from S3 — no artifact registration, no EU bucket writes."""
    with tempfile.NamedTemporaryFile(suffix=".vcf.gz") as tmp:
        tmp.write(vcf_uri.read_bytes())
        tmp.flush()
        vcf = pysam.VariantFile(tmp.name)
        data_records = []
        for record in vcf:
            rec_dict = {
                'CHROM': record.chrom, 'POS': record.pos, 'ID': record.id,
                'REF': record.ref,
                'ALT': ','.join(str(a) for a in record.alts) if record.alts else '.',
                'QUAL': record.qual,
                'FILTER': 'PASS' if list(record.filter.keys()) == ['PASS'] else ','.join(record.filter),
                'FORMAT': ':'.join(record.format),
            }
            for key in record.info.keys():
                try:
                    value = record.info[key]
                    rec_dict[f'INFO_{key}'] = value[0] if isinstance(value, tuple) and len(value) == 1 else value
                except TypeError:
                    pass
            if record.samples:
                sample_name = list(record.samples)[0]
                sample_data = record.samples[sample_name]
                for field in record.format:
                    try:
                        value = sample_data[field]
                        rec_dict[f'SAMPLE_{field}'] = '/'.join("." if v is None else str(v) for v in value) if isinstance(value, tuple) else str(value)
                    except (KeyError, TypeError):
                        rec_dict[f'SAMPLE_{field}'] = "."
                rec_dict["SAMPLE_NAME"] = sample_name
            data_records.append(rec_dict)
    vcf_df = pd.DataFrame(data_records)
    vcf_df["SAMPLE_SM"] = vcf_df["SAMPLE_SM"].astype(float)
    vcf_df["SAMPLE_CN"] = vcf_df["SAMPLE_CN"].astype(int)
    vcf_df["SAMPLE_BC"] = vcf_df["SAMPLE_BC"].astype(int)
    return vcf_df

def process_sample(sample):
    # Skip if already registered — idempotent
    key = f"data/dragen-3.7.6/hg38-graph-based/{sample}/{sample}.cnv.parquet"
    if ln.Artifact.filter(key=key).exists():
        return sample, None

    # Read directly from s3://1000genomes-dragen — never touches lamindb storage
    cnv_uri = upath / f"data/dragen-3.7.6/hg38-graph-based/{sample}/{sample}.cnv.vcf.gz"
    vcf_df = read_vcf(cnv_uri)

    parquet_art = ln.Artifact.from_dataframe(
        vcf_df, key=key, schema=schema,
    ).save()
    parquet_art.ulabels.add(cnv_label)
    return sample, len(vcf_df)

available_samples = [
    p.name for p in (upath / "data/dragen-3.7.6/hg38-graph-based/").iterdir()
    if (p / f"{p.name}.cnv.vcf.gz").exists()
]

done, skipped, failed = 0, 0, 0
with ThreadPoolExecutor(max_workers=32) as executor:
    futures = {executor.submit(process_sample, s): s for s in available_samples}
    for future in as_completed(futures):
        sample = futures[future]
        try:
            s, n = future.result()
            if n is None:
                skipped += 1
            else:
                done += 1
                print(f"✓ {s}: {n} rows  [{done} done, {skipped} skipped]")
        except Exception as e:
            failed += 1
            print(f"✗ {sample}: {e}")

print(f"Finished: {done} new, {skipped} skipped, {failed} failed")


→ connected lamindb: laminlabs/lakehouse-benchmarks
→ found notebook 1000genome_ingestion_conversion.ipynb, making new version
→ created Transform('dqmJBREmxYSV0003', key='1000genome_ingestion_conversion.ipynb'), started new Run('e1XtEb7mHnh8MoVj') at 2026-07-01 18:56:15 UTC
→ notebook imports: lamindb-core==2.6.1 pandas==2.3.3 pysam==0.24.0
→ returning ulabel with same name: 'CNV'


[E::idx_find_and_load] Could not retrieve index file for '/tmp/tmpowpkq9ug.vcf.gz'
[E::idx_find_and_load] Could not retrieve index file for '/tmp/tmplmwkcoe4.vcf.gz'
[E::idx_find_and_load] Could not retrieve index file for '/tmp/tmpn980d94m.vcf.gz'
[E::idx_find_and_load] Could not retrieve index file for '/tmp/tmp9h5fmvlu.vcf.gz'
[E::idx_find_and_load] Could not retrieve index file for '/tmp/tmp4y0x9kp4.vcf.gz'
[E::idx_find_and_load] Could not retrieve index file for '/tmp/tmpuds79jl8.vcf.gz'
[E::idx_find_and_load] Could not retrieve index file for '/tmp/tmpxcdbeekv.vcf.gz'
[E::idx_find_and_load] Could not retrieve index file for '/tmp/tmpdqho0eac.vcf.gz'[E::idx_find_and_load] Could not retrieve index file for '/tmp/tmpzjbfhrzd.vcf.gz'

[E::idx_find_and_load] Could not retrieve index file for '/tmp/tmpo_1tk7uw.vcf.gz'
[E::idx_find_and_load] Could not retrieve index file for '/tmp/tmp0_1pxb2t.vcf.gz'
[E::idx_find_and_load] Could not retrieve index file for '/tmp/tmpiutpthx2.vcf.gz'
[E::

→ loading artifact into memory for validation
→ loading artifact into memory for validation
→ loading artifact into memory for validation
→ loading artifact into memory for validation
→ loading artifact into memory for validation
→ loading artifact into memory for validation
→ loading artifact into memory for validation
→ loading artifact into memory for validation
→ loading artifact into memory for validation
→ loading artifact into memory for validation
→ loading artifact into memory for validation
→ loading artifact into memory for validation
→ loading artifact into memory for validation
→ loading artifact into memory for validation
→ loading artifact into memory for validation
→ loading artifact into memory for validation
→ loading artifact into memory for validation
→ loading artifact into memory for validation
→ loading artifact into memory for validation
→ loading artifact into memory for validation
→ loading artifact into memory for validation
→ loading artifact into memory for

[E::idx_find_and_load] Could not retrieve index file for '/tmp/tmpwbo7ldri.vcf.gz'
[E::idx_find_and_load] Could not retrieve index file for '/tmp/tmpvo_msqjq.vcf.gz'


→ returning schema with same hash: Schema(uid='0000000000000000', is_type=False, name=None, description=None, n_members=None, coerce=None, flexible=True, itype='Feature', otype=None, hash='kMi7B_N88uu-YnbTLDU-DA', minimal_set=True, ordered_set=False, maximal_set=False, branch_id=1, created_on_id=1, space_id=1, created_by_id=2, run_id=90, type_id=None, created_at=2026-06-19 21:05:55 UTC, is_locked=False)
→ returning schema with same hash: Schema(uid='0000000000000000', is_type=False, name=None, description=None, n_members=None, coerce=None, flexible=True, itype='Feature', otype=None, hash='kMi7B_N88uu-YnbTLDU-DA', minimal_set=True, ordered_set=False, maximal_set=False, branch_id=1, created_on_id=1, space_id=1, created_by_id=2, run_id=90, type_id=None, created_at=2026-06-19 21:05:55 UTC, is_locked=False)
→ returning schema with same hash: Schema(uid='0000000000000000', is_type=False, name=None, description=None, n_members=None, coerce=None, flexible=True, itype='Feature', otype=None, has

...

→ go to https://lamin.ai/laminlabs/lakehouse-benchmarks/artifact/XS1HuP4Ljdqg0AqB0000
→ loading artifact into memory for validation
→ loading artifact into memory for validation
✓ NA21101: 1489 rows  [3167 done, 0 skipped]
! no values were validated for columns!
! no values were validated for columns!
✓ NA21103: 1394 rows  [3168 done, 0 skipped]
... uploading BYEwqk4jleHVA8L20000.parquet: 100.0%
• replacing the existing cache path /home/sagemaker-user/.cache/lamindb/lamin-eu-central-1/MBiQHz7l46Jk/data/dragen-3.7.6/hg38-graph-based/NA21113/NA21113.cnv.parquet
→ loading artifact into memory for validation
→ loading artifact into memory for validation
... uploading nHOBGfgXcOyzI5fx0000.parquet: 100.0%
• replacing the existing cache path /home/sagemaker-user/.cache/lamindb/lamin-eu-central-1/MBiQHz7l46Jk/data/dragen-3.7.6/hg38-graph-based/NA21114/NA21114.cnv.parquet
✓ NA21104: 1433 rows  [3169 done, 0 skipped]
→ go to https://lamin.ai/laminlabs/lakehouse-benchmarks/artifact/cgd0oVRf98l9Xa

[E::idx_find_and_load] Could not retrieve index file for '/tmp/tmp_e75s26e.vcf.gz'


→ loading artifact into memory for validation
→ loading artifact into memory for validation
! no values were validated for columns!
... uploading NUp44htp9Zbitecw0000.parquet:  0.0%→ loading artifact into memory for validation


[E::idx_find_and_load] Could not retrieve index file for '/tmp/tmpukh2weo6.vcf.gz'
[E::idx_find_and_load] Could not retrieve index file for '/tmp/tmp8u1c3y6j.vcf.gz'
[E::idx_find_and_load] Could not retrieve index file for '/tmp/tmpzctb2z39.vcf.gz'


! no values were validated for columns!
... uploading bq0Org4GVAn046xC0000.parquet:  0.0%→ loading artifact into memory for validation
→ go to https://lamin.ai/laminlabs/lakehouse-benchmarks/artifact/Vje2ig1CyncaMj7J0000
✓ NA21105: 1561 rows  [3170 done, 0 skipped]
→ loading artifact into memory for validation
→ loading artifact into memory for validation
→ returning schema with same hash: Schema(uid='0000000000000000', is_type=False, name=None, description=None, n_members=None, coerce=None, flexible=True, itype='Feature', otype=None, hash='kMi7B_N88uu-YnbTLDU-DA', minimal_set=True, ordered_set=False, maximal_set=False, branch_id=1, created_on_id=1, space_id=1, created_by_id=2, run_id=90, type_id=None, created_at=2026-06-19 21:05:55 UTC, is_locked=False)
... uploading HTzFAPueu1XMSFbU0000.parquet:  0.0%✓ NA21106: 1658 rows  [3171 done, 0 skipped]
✓ NA21107: 1542 rows  [3172 done, 0 skipped]
! no values were validated for columns!
... uploading NUp44htp9Zbitecw0000.parquet: 100.0%
• rep

In [2]:
# --- Collect all 1000 Genomes parquet artifacts into a full collection ---
# Query once; key prefix covers every sample converted to parquet in this pipeline.
all_parquet_arts = list(
    ln.Artifact.filter(
        key__startswith="data/dragen-3.7.6/hg38-graph-based/",
        key__endswith=".cnv.parquet",
        suffix=".parquet",
    ).order_by("key").all()
)
print(f"Found {len(all_parquet_arts)} parquet artifacts")

# get_or_create pattern: if the collection already exists, revise it;
# otherwise create fresh. Avoids duplicate collections on re-runs.
existing_full = ln.Collection.filter(key="1000genomes_parquet_full").one_or_none()

ln.Collection(
    all_parquet_arts,
    key="1000genomes_parquet_full",
    description="All 1000 Genomes CNV parquet shards — full cohort",
    **(dict(revises=existing_full) if existing_full else {}),
).save()

print(f"Saved collection '1000genomes_parquet_full' with {len(all_parquet_arts)} artifacts")


Found 3201 parquet artifacts
Saved collection '1000genomes_parquet_full' with 3201 artifacts


In [3]:
ln.finish()

• please hit CTRL + s to save the notebook in your editor . ✓
→ finished Run('e1XtEb7mHnh8MoVj') after 16m at 2026-07-01 19:12:54 UTC
→ go to: https://lamin.ai/laminlabs/lakehouse-benchmarks/transform/dqmJBREmxYSV0003
→ to update your notebook from the CLI, run: lamin save /home/sagemaker-user/lakehouse-benchmarks/1000genome_ingestion_conversion.ipynb
